# Real Out-of-Sample Strategy Validation (Walk-Forward + Pre-Registered Holdout)

Epic 15: answers `README.md`'s Known Gap "Has the strategy shown a positive out-of-sample
(holdout) return on real data? ❌ Never run on real data" for the first time. This is more
foundational than Epic 13/14 (which validated risk-control *mechanisms* under crash stress) --
this is about whether the momentum *signal itself* generalizes, or whether the shipped
`lookback_period: 12` was just tuned to the specific historical sample it happened to be
picked from.

**A scaffold for this already existed** (`notebooks/research/
DHI0016_notebook1_research_and_EDA_IMPROVED.ipynb`'s cells 49-57, `core/
functions_quant_extensions.py`'s `pre_registered_split()`/`walk_forward_lookback_holding()`/
`bootstrap_sharpe_ci()`), fully coded but never actually executed against real data, the exact
same "written but never run" pattern Epic 12/13 already found and fixed elsewhere in this
project.

**A real design gap found in that scaffold**: its `_quick_backtest()` helper is a simplified
mean-monthly-return approximation (no regime filter, no volatility targeting, no stop-loss, no
slippage/commission), not this project's real engine, and it reimplements picks selection
without `.dropna()` before `.nsmallest()`, the exact real bug `get_top_etfs()`'s own docstring
documents and fixes elsewhere in this codebase. This notebook instead uses `core/
functions_quant_extensions.py`'s NEW `run_walk_forward_lookback_search()` (Epic 15), which wires
the walk-forward search through the REAL `core/strategy_signals.py`'s
`generate_strategy_monthly_picks()` + `backtest/momentum_backtest.py`'s `run_custom_backtest()`,
the same real pipeline Epic 13/14 already use, so this validates the strategy people would
actually run, not an approximation.

**Scope boundary, same as Epic 13**: most of `portfolio1`'s real tickers didn't exist back to
2005 (`ARM` IPO'd 2023, `PLTR` 2020), so this reuses Epic 13's already-cached long-history ETF
proxy universe (`crash_test_daily_prices.pkl`, 17 tickers, 2005-01-03 through today, no new
fetch needed) with `portfolio1`'s REAL `default_risk` config (`strategy_type=momentum`,
`top_n=5`, `stop_loss_pct=0.10`, regime filter, vol targeting, all unchanged) applied to it.
This validates whether the STRATEGY MECHANICS generalize out of sample on a representative
universe, not a claim about the exact current portfolio's own tickers.

Entirely backtest/research-only, no live paper account touched at all.

In [ ]:
# Package is pip-installed editable, no sys.path hacking needed
from dataclasses import replace

import pandas as pd

from momentum_trading.daily_runner import load_config
from momentum_trading.core import functions_quant_extensions as fnx
from momentum_trading.core.strategy_signals import generate_strategy_monthly_picks
from momentum_trading.backtest.momentum_backtest import run_custom_backtest

## 1. Load the real config and the cached proxy-universe price history

In [ ]:
config = load_config()
portfolio1_cfg = config["portfolios_resolved"]["portfolio1"]["cfg"]  # real monthly config

# Confirmed via a direct yfinance check (Epic 13) that all 17 have real data back to 2005.
PROXY_TICKERS = [
    "SPY", "QQQ", "DIA", "XLK", "XLF", "XLE", "XLI", "XLP", "XLU", "XLV", "XLY",
    "GLD", "TLT", "IEF", "SHY", "LQD", "IWM",
]

# Built by Epic 13's fetch step, relative to this notebook's own directory.
daily_prices = pd.read_pickle("crash_test_daily_prices.pkl")
print(daily_prices.shape, daily_prices.index.min(), "->", daily_prices.index.max())
print(f"strategy_type={portfolio1_cfg.strategy_type} top_n={portfolio1_cfg.top_n} "
      f"holding_period={portfolio1_cfg.holding_period} lookback_period={portfolio1_cfg.lookback_period} "
      f"stop_loss_pct={portfolio1_cfg.stop_loss_pct} use_regime_filter={portfolio1_cfg.use_regime_filter}")

## 2. Pre-registered train / holdout split

Commit to this split BEFORE any tuning. `train` is used for walk-forward parameter search
below; `holdout` is not touched (not even glanced at) until the single final evaluation.

In [ ]:
train, holdout = fnx.pre_registered_split(daily_prices, split_date="2015-01-01")
print(f"Train:   {train.index.min().date()} to {train.index.max().date()} ({len(train)} rows)")
print(f"Holdout: {holdout.index.min().date()} to {holdout.index.max().date()} ({len(holdout)} rows)")

## 3. Walk-forward lookback search, TRAIN only

Real-engine walk-forward search (Epic 15's `run_walk_forward_lookback_search()`): for each
rolling fold within `train`, pick the best `lookback_period` from the grid below using ONLY
that fold's own training slice (via the real `generate_strategy_monthly_picks()` +
`run_custom_backtest()` pipeline, `portfolio1`'s real config otherwise unchanged), then evaluate
that choice on the immediately following, unseen test slice. A real edge shows `test_Sharpe`
reasonably close to `train_Sharpe` across folds; a large, consistent gap is the overfitting
signature.

In [ ]:
LOOKBACK_CANDIDATES = [6, 9, 12, 15, 18]  # months

wf_results = fnx.run_walk_forward_lookback_search(
    train, PROXY_TICKERS, portfolio1_cfg,
    lookback_candidates=LOOKBACK_CANDIDATES,
    train_years=4, test_years=1, step_years=1,
    metric="Sharpe",
)
pd.set_option("display.max_columns", None)
print(wf_results.to_string())

## 4. Choose the most robust lookback, evaluate on HOLDOUT exactly once

The most FREQUENTLY chosen lookback across folds (not just the single highest test_Sharpe
fold, which would just be cherry-picking one lucky fold), evaluated on `holdout` ONE time.
This is the project's first genuine out-of-sample number, report it as-is.

In [ ]:
chosen_lookback = int(wf_results["chosen_lookback"].mode().iloc[0])
print(f"Most frequently chosen lookback across {len(wf_results)} folds: {chosen_lookback} months")
print(f"Mean train_Sharpe across folds: {wf_results['train_Sharpe'].mean():.2f}")
print(f"Mean test_Sharpe across folds:  {wf_results['test_Sharpe'].mean():.2f}")

holdout_cfg = replace(portfolio1_cfg, lookback_period=chosen_lookback)
holdout_start = holdout.index.min()

# Bounded only at the end (Epic 14's own lesson): use the FULL daily_prices panel for real
# lookback history before holdout_start, filter the resulting PICKS down to the holdout window.
picks_full = generate_strategy_monthly_picks(
    daily_prices, PROXY_TICKERS, holdout_cfg, chosen_lookback, holdout_cfg.top_n,
)
picks_holdout = picks_full[picks_full.index >= holdout_start]

holdout_bt = run_custom_backtest(picks_holdout, daily_prices, **holdout_cfg.__dict__)
holdout_tearsheet = holdout_bt.attrs.get("tearsheet", {})

print(f"\nHOLDOUT ({holdout.index.min().date()} to {holdout.index.max().date()}), "
      f"chosen_lookback={chosen_lookback}, reported ONCE:")
for k, v in holdout_tearsheet.items():
    print(f"  {k}: {v}")

## 5. Block-bootstrap confidence interval on the holdout Sharpe

A confidence interval, not just a single point estimate from one historical path.

In [ ]:
holdout_monthly_returns = holdout_bt["Portfolio Monthly Return"].dropna()

ci = fnx.bootstrap_sharpe_ci(holdout_monthly_returns, n_bootstrap=2000, block_size=6)
print(f"Holdout Sharpe point estimate: {ci['point_estimate']:.2f}")
print(f"{int(ci['confidence']*100)}% CI: [{ci['ci_low']:.2f}, {ci['ci_high']:.2f}]")
print(f"% of bootstrap samples with positive Sharpe: {ci['pct_bootstrap_samples_positive']*100:.1f}%")